# SportSRC API Data Importer

Base URL: `https://api.sportsrc.org/`  
Auth: None required (free tier, 20 RPS limit)  
All responses: JSON

In [2]:
import requests
import pandas as pd
from pprint import pprint

BASE_URL = "https://api.sportsrc.org/"

## 1. Sports Categories

`GET /?data=sports` — returns available sport categories (Football, Basketball, UFC, etc.)

In [3]:
def get_sports() -> list[dict]:
    resp = requests.get(BASE_URL, params={"data": "sports"})
    resp.raise_for_status()
    return resp.json()

sports = get_sports()
pprint(sports)

{'data': [{'id': 'olympics', 'name': 'Olympics'},
          {'id': 'basketball', 'name': 'Basketball'},
          {'id': 'football', 'name': 'Football'},
          {'id': 'american-football', 'name': 'American Football'},
          {'id': 'hockey', 'name': 'Hockey'},
          {'id': 'baseball', 'name': 'Baseball'},
          {'id': 'motor-sports', 'name': 'Motor Sports'},
          {'id': 'fight', 'name': 'Fight (UFC, Boxing)'},
          {'id': 'tennis', 'name': 'Tennis'},
          {'id': 'rugby', 'name': 'Rugby'},
          {'id': 'golf', 'name': 'Golf'},
          {'id': 'billiards', 'name': 'Billiards'},
          {'id': 'afl', 'name': 'AFL'},
          {'id': 'darts', 'name': 'Darts'},
          {'id': 'cricket', 'name': 'Cricket'},
          {'id': 'other', 'name': 'Other'}],
 'success': True}


## 2. Match Schedules

`GET /?data=matches&category={category}` — upcoming matches for a sport.  
Set `CATEGORY` to any value returned above (e.g. `"football"`, `"basketball"`).

In [4]:
CATEGORY = "basketball"  # <-- change me

def get_matches(category: str) -> list[dict]:
    resp = requests.get(BASE_URL, params={"data": "matches", "category": category})
    resp.raise_for_status()
    return resp.json()

matches = get_matches(CATEGORY)
matches_df = pd.DataFrame(matches)
matches_df.head()

,success,data
0,True,"{'id': 'nflstreams_live', 'title': 'NFL Stream..."
1,True,{'id': 'live_ncaa_se-louisiana-texas-am-cc-liv...
2,True,{'id': 'live_ncaa_northwestern-st-incarnate-wo...
3,True,{'id': 'live_ncaa_mcneese-st-utrgv-live-stream...
4,True,{'id': 'detroit-pistons-vs-san-antonio-spurs-2...


## 3. Match Detail & Stream

`GET /?data=detail&category={category}&id={match-id}` — full details + stream embed for a single match.  
Set `MATCH_ID` to an `id` value from the schedules response above.

In [ ]:
MATCH_ID = ""  # <-- paste a match id from matches_df

def get_match_detail(category: str, match_id: str) -> dict:
    resp = requests.get(BASE_URL, params={"data": "detail", "category": category, "id": match_id})
    resp.raise_for_status()
    return resp.json()

if MATCH_ID:
    detail = get_match_detail(CATEGORY, MATCH_ID)
    pprint(detail)
else:
    print("Set MATCH_ID to a value from matches_df above.")

## 4. Results, Tables & Leagues

`GET /?data=results&category={leagues|tables|scores}&league={code}`

| `category` | `league` required? | Description |
|---|---|---|
| `leagues` | No | List all available leagues |
| `tables` | Yes | Standings for a league (e.g. `PL`) |
| `scores` | Yes | Historical scores for a league |

In [5]:
def get_results(category: str, league: str | None = None) -> list[dict] | dict:
    params = {"data": "results", "category": category}
    if league:
        params["league"] = league
    resp = requests.get(BASE_URL, params=params)
    resp.raise_for_status()
    return resp.json()

# --- League listing ---
leagues = get_results("leagues")
leagues_df = pd.DataFrame(leagues)
print("Leagues:")
leagues_df.head()

Leagues:


,success,data
0,True,"{'id': 'WC', 'name': 'World Cup'}"
1,True,"{'id': 'CL', 'name': 'UEFA Champions League'}"
2,True,"{'id': 'BL1', 'name': 'Bundesliga'}"
3,True,"{'id': 'DED', 'name': 'Eredivisie'}"
4,True,"{'id': 'BSA', 'name': 'Campeonato Brasileiro S..."


In [6]:
LEAGUE_CODE = "PL"  # <-- change to a code from leagues_df

# --- Standings ---
tables = get_results("tables", league=LEAGUE_CODE)
tables_df = pd.DataFrame(tables)
print(f"Standings — {LEAGUE_CODE}:")
tables_df.head()

Standings — PL:


,success,data
filters,True,{'season': '2025'}
area,True,"{'id': 2072, 'name': 'England', 'code': 'ENG',..."
competition,True,"{'id': 2021, 'name': 'Premier League', 'code':..."
season,True,"{'id': 2403, 'startDate': '2025-08-15', 'endDa..."
standings,True,"[{'stage': 'REGULAR_SEASON', 'type': 'TOTAL', ..."


In [7]:
# --- Historical scores ---
scores = get_results("scores", league=LEAGUE_CODE)
scores_df = pd.DataFrame(scores)
print(f"Scores — {LEAGUE_CODE}:")
scores_df.head()

Scores — PL:


,success,data
live,True,[]
finished,True,[]
last_updated,True,2026-02-23T15:50:21+00:00
